# Fortune Advisory — Pull BCTC từ vnstock
**Kéo IS + BS + CF của toàn bộ cổ phiếu niêm yết (KBS), 4 quý gần nhất.**

Notebook gồm 8 cell — chạy tuần tự từ đầu đến cuối.  
Trước khi chạy: chỉnh `OUTPUT_DIR` trong **Cell 3**.

In [1]:
import sys

# Install dependencies
get_ipython().system(f'{sys.executable} -m pip install vnstock pandas openpyxl tqdm -q')

# KBS URL fix (required for Finance API to work)
import importlib, pathlib
_const = pathlib.Path(sys.prefix) / 'Lib/site-packages/vnstock/explorer/kbs/const.py'
if _const.exists():
    src    = _const.read_text(encoding='utf-8')
    broken = "_SAS_FINANCE_INFO_URL = f'{_SAS_STOCK_URL}/finance-info'"
    fixed  = "_SAS_FINANCE_INFO_URL = f'{_IIS_BASE_URL}/stock/finance-info'"
    if broken in src:
        _const.write_text(src.replace(broken, fixed), encoding='utf-8')
        print('Applied KBS URL fix')
    else:
        print('KBS URL already OK')
else:
    print('WARN: const.py not found — skip URL fix')

KBS URL already OK



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, time, warnings
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from vnstock import Finance, Vnstock, register_user

warnings.filterwarnings('ignore')

# Register API key (follow the prompt, or pass key directly: register_user('YOUR_KEY'))
register_user()

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 **Vnstock 4.0.2 is available**
Current: 3.5.1 (Python 3.13)
Update: `pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.8 is available**
Current: 2.4.6 (Python 3.13)
Update: `pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history


  VNSTOCK - ĐĂNG KÝ API KEY
✓ API key: vnstock_b3154f4...
✓ Tier (Gói): free
✓ Giới hạn (Limits): {'per_minute': 60, 'per_hour': 3600}

✓ API key: vnst***4...
✓ Tier (Gói): free
✓ Giới hạn (Limits): {'per_minute': 60, 'per_hour': 3600}


True

In [ ]:
# ── ONLY EDIT THIS CELL IF NEEDED ────────────────────────────────────
# Default '.' saves generated files in the same folder as this notebook.
# Change to a full path if you want output elsewhere.
OUTPUT_DIR = '.'
# ──────────────────────────────────────────────────────────────────────

DELAY_SEC  = 1.5   # seconds between tickers (free tier: 60 req/min)
N_QUARTERS = 4     # most recent quarters to keep

print(f'Output : {os.path.abspath(OUTPUT_DIR)}')
print(f'Delay  : {DELAY_SEC}s | Quarters: {N_QUARTERS}')

Output : c:\Users\Lenovo\Downloads\files
Delay  : 1.5s | Quarters: 8


In [9]:
listing      = Vnstock().stock(symbol='ACB', source='KBS').listing

# All listed stocks (exclude ETF, CW, bonds, futures)
all_df       = listing.symbols_by_exchange()
all_df       = all_df[all_df['type'] == 'stock'].copy()

# Exchange map (from KBS listing — reliable)
exchange_map = all_df.set_index('symbol')['exchange'].to_dict()
TARGET       = all_df['symbol'].tolist()
dist         = all_df.groupby('exchange').size().to_dict()
print(f'{len(TARGET)} tickers: {dist}')
print(f'Estimated run time: ~{len(TARGET) * DELAY_SEC / 3600:.1f} hours at {DELAY_SEC}s/ticker')

# ── Industry map from SSI export ──────────────────────────────────────
# File: ssi_industry.xlsx  |  Sheet: TỔNG QUAN  |  Cols: MÃ, NGÀNH
# Place this file in the same folder as this notebook (not OUTPUT_DIR).
SSI_FILE = 'ssi_industry.xlsx'  # same folder as this notebook

if os.path.exists(SSI_FILE):
    df_ssi       = pd.read_excel(SSI_FILE, sheet_name='TỔNG QUAN', dtype=str)
    df_ssi.columns = df_ssi.columns.str.strip()
    df_ssi['MÃ']   = df_ssi['MÃ'].str.strip().str.upper()
    industry_map   = df_ssi.set_index('MÃ')['NGÀNH'].to_dict()
    print(f'Industry map: {len(industry_map)} tickers from ssi_industry.xlsx')
else:
    industry_map = {}
    print('WARN: ssi_industry.xlsx not found — industry column will be empty.')
    print('      Place ssi_industry.xlsx in the same folder as this notebook and re-run.')

1538 tickers: {'HNX': 302, 'HOSE': 402, 'UPCOM': 834}
Estimated run time: ~0.6 hours at 1.5s/ticker
Industry map: 1537 tickers from ssi_industry.xlsx


In [5]:
def flatten_multiindex(df):
    """Flatten MultiIndex columns to single-level strings."""
    if df is None or df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [
            '_'.join(str(l) for l in c if str(l) not in ('', 'nan')).strip('_')
            for c in df.columns
        ]
    return df


def parse_quarter_col(col):
    """Parse a column name like '2024-Q3' or 'Q3/2024' into (year, quarter)."""
    s = str(col).strip()
    m = re.match(r'^(\d{4})[^\d]*(Q|q)?\s*([1-4])$', s, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(3))
    m = re.match(r'^(Q|q)\s*([1-4])[^\d]*(\d{4})$', s, re.IGNORECASE)
    if m:
        return int(m.group(3)), int(m.group(2))
    return None, None


def last_completed_quarter():
    """Return (year, quarter) of the most recently completed quarter."""
    now = datetime.now()
    q   = (now.month - 1) // 3 + 1
    return (now.year - 1, 4) if q == 1 else (now.year, q - 1)


def fetch_one(symbol, report_type):
    """
    Fetch one financial statement from KBS.
    Returns raw DataFrame (rows = line items, cols = quarter periods), or None.
    """
    try:
        fin    = Finance(source='KBS', symbol=symbol, get_all=True, show_log=False)
        method = getattr(fin, report_type)
        df     = method(period='quarter')
        if df is None or df.empty:
            return None
        df = flatten_multiindex(df)
        df.columns = [str(c).strip() for c in df.columns]
        df.insert(0, 'ticker', symbol)
        return df
    except Exception:
        return None


def to_wide(df):
    """
    Pivot raw fetch output into tidy format:
        rows = one per ticker × quarter
        cols = financial line items
    """
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Identify period columns (e.g. '2024-Q3')
    max_y, max_q = last_completed_quarter()
    pcols = [
        c for c in df.columns
        if (lambda y, q: y and q and y * 100 + q <= max_y * 100 + max_q)(*parse_quarter_col(c))
    ]
    if not pcols:
        return pd.DataFrame()

    # Find item-name column
    known    = ('item', 'item_en', 'item_id', 'Chi tieu', 'chi_tieu', 'name', 'indicator')
    item_col = next((c for c in known if c in df.columns), None)
    if item_col is None:
        period_set = set(pcols)
        item_col   = next(
            (c for c in df.columns
             if c != 'ticker' and c not in period_set and df[c].dtype == object),
            None
        )
    if item_col is None:
        return pd.DataFrame()

    # Melt → add year/quarter → pivot to wide
    melted = df[['ticker', item_col] + pcols].melt(
        id_vars=['ticker', item_col], var_name='pcol', value_name='value'
    )
    parsed  = melted['pcol'].apply(
        lambda x: pd.Series(parse_quarter_col(x), index=['year', 'quarter'])
    )
    melted  = pd.concat([melted.drop(columns='pcol'), parsed], axis=1)
    melted  = melted.dropna(subset=['year', 'quarter'])
    melted  = melted[melted['quarter'] != 0]        # drop any annual rows
    melted['year']    = melted['year'].astype(int)
    melted['quarter'] = melted['quarter'].astype(int)

    wide = melted.pivot_table(
        index=['ticker', 'year', 'quarter'],
        columns=item_col,
        values='value',
        aggfunc='first'
    ).reset_index()
    wide.columns.name = None
    return wide.sort_values(['ticker', 'year', 'quarter']).reset_index(drop=True)


print('Functions ready.')

Functions ready.


In [6]:
# Quick test with 1 ticker — verify the pipeline before running the full batch
test_sym = TARGET[0]
print(f'Testing: {test_sym}\n')

raw  = fetch_one(test_sym, 'income_statement')
wide = to_wide(raw)

if wide.empty:
    print('FAILED — check API connection or run register_user() again')
else:
    quarters = sorted(
        wide[['year','quarter']].apply(lambda r: f"{r.year}-Q{r.quarter}", axis=1)
    )
    print(f'Shape      : {wide.shape[0]} rows x {wide.shape[1]} cols')
    print(f'Quarters   : {quarters}')
    print(f'First cols : {list(wide.columns[:8])}')
    print('\nOK — pipeline working, ready to run Cell 7')

Testing: TCB



Shape      : 4 rows x 26 cols
Quarters   : ['2025-Q2', '2025-Q3', '2025-Q4', '2026-Q1']
First cols : ['ticker', 'year', 'quarter', '1. Thu nhập lãi và các khoản thu nhập tương tự', '2. Chi phí lãi và các chi phí tương tự', '3. Thu nhập từ hoạt động dịch vụ', '4. Chi phí hoạt động dịch vụ', '5. Thu nhập từ hoạt động khác']

OK — pipeline working, ready to run Cell 7


In [7]:
# ── Pull IS + BS + CF for all tickers ────────────────────────────────
# Estimated run time is shown in Cell 4.
# If interrupted, re-run this cell from scratch (frames will reset).

KEY = ['ticker', 'exchange', 'year', 'quarter']
frames_is, frames_bs, frames_cf = [], [], []
n_ok, n_fail = 0, 0

for sym in tqdm(TARGET, desc='Pulling BCTC'):
    exch   = exchange_map.get(sym, '')
    got_any = False

    for rtype, frames in [
        ('income_statement', frames_is),
        ('balance_sheet',    frames_bs),
        ('cash_flow',        frames_cf),
    ]:
        raw  = fetch_one(sym, rtype)
        wide = to_wide(raw)
        if not wide.empty:
            wide.insert(1, 'exchange', exch)
            frames.append(wide)
            got_any = True

    if got_any: n_ok   += 1
    else:       n_fail += 1

    time.sleep(DELAY_SEC)

print(f'\nDone  : {n_ok} tickers OK | {n_fail} skipped (no data)')
print(f'Frames: IS={len(frames_is)}, BS={len(frames_bs)}, CF={len(frames_cf)}')

Pulling BCTC:   0%|          | 0/1538 [00:00<?, ?it/s]

Pulling BCTC: 100%|██████████| 1538/1538 [2:34:36<00:00,  6.03s/it]  


Done  : 1338 tickers OK | 200 skipped (no data)
Frames: IS=1334, BS=1337, CF=982


In [10]:
# ── Filter to N most recent quarters + export ────────────────────────

def filter_recent_quarters(df, n=N_QUARTERS):
    """Keep only the n most recently completed quarters with >= 5% ticker coverage."""
    if df.empty:
        return df
    max_y, max_q = last_completed_quarter()
    total  = max(df['ticker'].nunique(), 1)
    counts = (df.groupby(['year', 'quarter'])['ticker'].nunique()
                .reset_index(name='cnt'))
    counts = counts[(counts['year'] * 100 + counts['quarter']) <= (max_y * 100 + max_q)]
    counts = counts[counts['cnt'] / total >= 0.05]
    counts = counts.sort_values(['year', 'quarter'], ascending=False).head(n)
    keep   = set(zip(counts['year'].astype(int), counts['quarter'].astype(int)))
    mask   = df.apply(lambda r: (int(r['year']), int(r['quarter'])) in keep, axis=1)
    return df[mask].reset_index(drop=True)


# Concatenate + filter
df_is = filter_recent_quarters(pd.concat(frames_is, ignore_index=True)) if frames_is else pd.DataFrame()
df_bs = filter_recent_quarters(pd.concat(frames_bs, ignore_index=True)) if frames_bs else pd.DataFrame()
df_cf = filter_recent_quarters(pd.concat(frames_cf, ignore_index=True)) if frames_cf else pd.DataFrame()

for name, d in [('KQKD', df_is), ('CDKT', df_bs), ('LCTT', df_cf)]:
    if not d.empty:
        q_range = f"{d['year'].min()}-Q{d.loc[d['year']==d['year'].min(),'quarter'].min()} -> " \
                  f"{d['year'].max()}-Q{d.loc[d['year']==d['year'].max(),'quarter'].max()}"
        print(f'{name}: {d["ticker"].nunique()} tickers | {q_range} | {d.shape}')
    else:
        print(f'{name}: EMPTY')

# Merge IS + BS + CF (outer join on KEY)
active = [f for f in [df_is, df_bs, df_cf] if not f.empty]
if not active:
    print('\nERROR: no data — re-run Cell 7')
else:
    df_merged = active[0]
    for f in active[1:]:
        df_merged = df_merged.merge(f, on=KEY, how='outer')

    # ── Industry column ───────────────────────────────────────────────────────
    # Normalize ticker case/whitespace before lookup
    ticker_norm = df_merged['ticker'].str.strip().str.upper()
    df_merged.insert(2, 'industry', ticker_norm.map(industry_map))

    n_total    = len(df_merged['ticker'].unique())
    n_mapped   = df_merged[df_merged['industry'].notna()]['ticker'].nunique()
    n_missing  = n_total - n_mapped
    pct        = n_mapped / n_total * 100 if n_total else 0

    if not industry_map:
        print('\nWARN: industry column is entirely empty.')
        print('      Industry Screener and benchmarking views will not work in Streamlit.')
        print('      → Place ssi_industry.xlsx in the notebook folder and re-run.')
    elif n_missing > 0:
        print(f'\nIndustry coverage: {n_mapped}/{n_total} tickers mapped ({pct:.0f}%)')
        print(f'  {n_missing} tickers have no industry — likely newly listed or not in SSI export.')
        print('  → Update ssi_industry.xlsx and re-run if coverage is low.')
    else:
        print(f'\nIndustry coverage: {n_mapped}/{n_total} tickers ({pct:.0f}%) — OK')

    df_merged = df_merged.sort_values(['exchange', 'ticker', 'year', 'quarter']).reset_index(drop=True)
    print(f'Merged: {df_merged.shape[0]} rows x {df_merged.shape[1]} cols')

    # Export to Excel
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts    = datetime.now().strftime('%Y%m%d_%H%M')
    fname = f'ALL_BCTC_KBS_quarter_{N_QUARTERS}Q_{ts}.xlsx'
    path  = os.path.join(OUTPUT_DIR, fname)

    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        df_merged.to_excel(writer, sheet_name='TONG_HOP', index=False)
        if not df_is.empty: df_is.to_excel(writer, sheet_name='KQKD', index=False)
        if not df_bs.empty: df_bs.to_excel(writer, sheet_name='CDKT', index=False)
        if not df_cf.empty: df_cf.to_excel(writer, sheet_name='LCTT', index=False)

    print(f'Saved : {path}')
    print(f'\nNext step: open streamlit_app.py and run the dashboard.')


KQKD: 1147 tickers | 2025-Q1 -> 2026-Q1 | (4524, 193)
CDKT: 1147 tickers | 2025-Q1 -> 2026-Q1 | (4526, 408)
LCTT: 899 tickers | 2025-Q1 -> 2026-Q1 | (1441, 197)

Industry coverage: 1147/1147 tickers (100%) — OK
Merged: 4526 rows x 791 cols
Saved : .\ALL_BCTC_KBS_quarter_8Q_20260504_1243.xlsx

Next step: open streamlit_app.py and run the dashboard.
